# Imports & Environment Setup

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
import tensorflow_datasets as tfds
from tensorflow.keras.models import Sequential

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Dataset Download (KaggleHub)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mahmoudreda55/satellite-image-classification")

print("Path to dataset files:", path)

# Load Dataset + Train/Validation/Test Split

In [ ]:
from IPython.testing import test
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
import os

base_dir_path = os.path.join(path, "data")

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.25,
    rotation_range=15,
    horizontal_flip=True,
    vertical_flip=True
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

image_size = 224
BATCH_SIZE = 32

train_ds = train_datagen.flow_from_directory(
    base_dir_path,
    subset="training",
    color_mode="rgb",
    shuffle=True,
    seed=42,
    target_size=(image_size, image_size),
    batch_size=BATCH_SIZE
)

val_ds = train_datagen.flow_from_directory(
    base_dir_path,
    subset="validation",
    color_mode="rgb",
    shuffle=False,
    seed=42,
    target_size=(image_size, image_size),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_ds = test_datagen.flow_from_directory(
    base_dir_path,
    color_mode="rgb",
    shuffle=True,
    seed=42,
    target_size=(image_size, image_size),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Class Distribution Check (Imbalance Detection)

In [ ]:
#check for imbalance
class_counts = train_ds.classes
unique_classes, class_counts = np.unique(class_counts, return_counts=True)
class_labels = list(train_ds.class_indices.keys())
class_distribution = dict(zip(class_labels, class_counts))
class_distribution

# Build the Model (ResNet50 Transfer Learning)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


base_model = ResNet50(weights='imagenet', include_top=False,
                      input_shape=(image_size, image_size, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

# Conservative training
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

# Model Summary

In [ ]:
model.summary()

# Train the Model

In [ ]:
hist = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs = 8,
    callbacks=callbacks
)

# Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print('Test accuracy:', test_acc)

# Model Preformance Visualization

In [ ]:
plt.plot(hist.history['accuracy'], label = 'accuracy')
plt.plot(hist.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend()
plt.show()

In [ ]:
plt.plot(hist.history['loss'], label = 'loss')
plt.plot(hist.history['val_loss'], label = 'val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Predictions Visualization

In [ ]:
# Simple version - just show 12 random predictions
test_ds.reset()
x, y = next(test_ds)
preds = model.predict(x[:12], verbose=0)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i, ax in enumerate(axes.ravel()):
    img = denormalize_image(x[i])
    actual = class_names[np.argmax(y[i])]
    pred = class_names[np.argmax(preds[i])]
    conf = preds[i].max() * 100

    ax.imshow(img)
    ax.axis('off')
    color = 'green' if actual == pred else 'red'
    ax.set_title(f'Actual:{actual}\nPredicted:{pred}\n{conf:.0f}%', color=color, size=9)
plt.tight_layout()
plt.show()